In [ ]:
import nltk
from nltk.corpus import wordnet as wn
import pandas as pd
from scipy.stats import spearmanr, pearsonr
import csv
import numpy as np

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
data = pd.read_csv('WordSim353.csv')

In [ ]:
data.head()

In [ ]:
def wup_similarity_by_hand(x, y):
    if x.lowest_common_hypernyms(y):
        return (2 * x.lowest_common_hypernyms(y)[0].min_depth()) / (x.min_depth() + y.min_depth())
    return 0

def path_similarity_by_hand(x, y):
    p = x.shortest_path_distance(y)
    d = x.max_depth()
    return (2 * d) - p if p is not None else 0

def lch_similarity_by_hand(x, y):
    p = x.shortest_path_distance(y)
    d = x.max_depth() + 1
    return -np.log((p+1)/(2*d)) if p is not None else 0


def path_similarity(x,y):
    return x.path_similarity(y) if x.path_similarity(y) is not None else 0

def lch_similarity(x,y):
    try:
        return x.lch_similarity(y)
    except:
        return 0

def wup_similarity(x,y):
    return x.wup_similarity(y) if x.wup_similarity(y) is not None else 0

In [ ]:
def get_similarity(word1: str, word2: str, sim_fn):
    synset1 = wn.synsets(word1)
    synset2 = wn.synsets(word2)

    if not synset1 or not synset2:
        return 0

    return max([sim_fn(x, y) for x in synset1 for y in synset2])

In [ ]:
sets_of_sim_metrics = {
    'metrics_by_hand': [
        path_similarity_by_hand,
        wup_similarity_by_hand,
        lch_similarity_by_hand
    ],
    'lib_metrics': [
        path_similarity,
        wup_similarity,
        lch_similarity
    ]
}


for id, metrics in sets_of_sim_metrics.items():

    print(f"===== {id.upper()} =====")

    results = {'path': [], 'wup': [], 'lch': []}

    for _, row in data.iterrows():
        word1, word2 = row['Word 1'], row['Word 2']

        for name, fn in zip(results.keys(), metrics):
            results[name].append(get_similarity(word1, word2, fn))

    for method in results.keys():
        spearman_corr, _ = spearmanr(data['Human (mean)'], results[method])
        pearson_corr, _ = pearsonr(data['Human (mean)'], results[method])

        print(f"{method.upper()} - Spearman: {spearman_corr:.3f}, Pearson: {pearson_corr:.3f}")

    print()

===== METRICS_BY_HAND =====
PATH - Spearman: 0.232, Pearson: 0.227
WUP - Spearman: 0.329, Pearson: 0.285
LCH - Spearman: 0.313, Pearson: 0.356

===== LIB_METRICS =====
PATH - Spearman: 0.294, Pearson: 0.364
WUP - Spearman: 0.339, Pearson: 0.297
LCH - Spearman: 0.301, Pearson: 0.326

